# exp_017 V18 Error Report

Objective:
- build a pooled validation report for the strongest native branch `exp_011`
- summarize weak and strong regions by class, taxon, site, and hour
- optionally crosswalk those results against attached V18 artifact metadata from `exp_015d`-style datasets

Success criteria:
- produce reusable CSV reports under `experiments/outputs/exp_017_v18_error_report`
- identify the weakest soundscape classes and the most difficult sites/hours
- if V18 artifacts are attached, summarize threshold/calibration behavior next to native classwise quality


In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score


def resolve_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'PROJECT_STATE.md').exists() and (candidate / 'experiments').exists():
            return candidate
    raise FileNotFoundError('Could not resolve repository root from current working directory')


REPO_ROOT = resolve_repo_root()
DATA_ROOT = REPO_ROOT / 'data' / 'birdclef-2026'
EXP011_ROOT = REPO_ROOT / 'experiments' / 'outputs' / 'exp_011_hgnetv2_soundscape_supervised'
OUTPUT_DIR = REPO_ROOT / 'experiments' / 'outputs' / 'exp_017_v18_error_report'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_SUB_PATH = DATA_ROOT / 'sample_submission.csv'
TAXONOMY_PATH = DATA_ROOT / 'taxonomy.csv'

# Optional: attach a V18 artifact dataset locally or on Kaggle and set a hint.
V18_ARTIFACTS_HINT = None  # e.g. 'birdclef-exp015c-v18-artifacts'
KAGGLE_INPUT_ROOT = Path('/kaggle/input')

print('repo_root:', REPO_ROOT)
print('exp011_root exists:', EXP011_ROOT.exists())
print('output_dir:', OUTPUT_DIR)
print('kaggle_input_exists:', KAGGLE_INPUT_ROOT.exists())


In [ ]:
FNAME_RE = re.compile(r'BC2026_(?:Train|Test)_(?P<file_id>[^_]+)_(?P<site>S\d+)_(?P<date>\d{8})_(?P<time>\d{6})\.ogg')
TEXTURE_TAXA = {'Amphibia', 'Insecta'}


def parse_soundscape_filename(name: str) -> dict:
    m = FNAME_RE.match(str(name))
    if not m:
        return {
            'file_id': None,
            'site': None,
            'date': pd.NaT,
            'time_utc': None,
            'hour_utc': np.nan,
        }
    d = m.groupdict()
    dt = pd.to_datetime(d['date'], format='%Y%m%d', errors='coerce')
    return {
        'file_id': d['file_id'],
        'site': d['site'],
        'date': dt,
        'time_utc': d['time'],
        'hour_utc': int(d['time'][:2]),
    }


def split_label_string(value: object) -> list[str]:
    if pd.isna(value):
        return []
    return [part.strip() for part in str(value).split(';') if part.strip()]


def safe_auc(y_true: np.ndarray, y_score: np.ndarray) -> float:
    positives = int(y_true.sum())
    negatives = int((1 - y_true).sum())
    if positives == 0 or negatives == 0:
        return np.nan
    try:
        return float(roc_auc_score(y_true, y_score))
    except Exception:
        return np.nan


def safe_classwise_auc(labels: np.ndarray, probs: np.ndarray, class_names: list[str]) -> pd.DataFrame:
    rows = []
    for idx, class_name in enumerate(class_names):
        y_true = labels[:, idx].astype(np.float32)
        y_prob = probs[:, idx].astype(np.float32)
        positives = int(y_true.sum())
        negatives = int(len(y_true) - positives)
        rows.append({
            'primary_label': class_name,
            'auc': safe_auc(y_true, y_prob),
            'positive_rows': positives,
            'negative_rows': negatives,
            'scored': positives > 0 and negatives > 0,
            'mean_prob': float(y_prob.mean()),
        })
    return pd.DataFrame(rows)


def grouped_macro_auc(meta_df: pd.DataFrame, labels: np.ndarray, probs: np.ndarray, group_col: str) -> pd.DataFrame:
    records = []
    series = meta_df[group_col].reset_index(drop=True)
    for group_value in series.dropna().sort_values().unique().tolist():
        idx = np.where(series.to_numpy() == group_value)[0]
        if len(idx) == 0:
            continue
        auc_df = safe_classwise_auc(labels[idx], probs[idx], PRIMARY_LABELS)
        scored = auc_df[auc_df['scored'] & auc_df['auc'].notna()]
        file_count = int(meta_df.iloc[idx]['filename'].nunique()) if 'filename' in meta_df.columns else np.nan
        records.append({
            group_col: group_value,
            'rows': int(len(idx)),
            'files': file_count,
            'scored_classes': int(len(scored)),
            'macro_auc': float(scored['auc'].mean()) if len(scored) else np.nan,
        })
    return pd.DataFrame(records).sort_values('macro_auc').reset_index(drop=True)


def load_exp011_pooled(root: Path) -> tuple[pd.DataFrame, np.ndarray, np.ndarray, np.ndarray]:
    fold_frames = []
    probs_parts = []
    logits_parts = []
    label_parts = []
    for fold_dir in sorted(root.glob('fold_*')):
        meta_path = fold_dir / 'best_valid_meta.csv'
        npz_path = fold_dir / 'best_valid_outputs.npz'
        if not meta_path.exists() or not npz_path.exists():
            continue
        meta = pd.read_csv(meta_path)
        arr = np.load(npz_path)
        if len(meta) != arr['labels'].shape[0]:
            raise ValueError(f'meta/output row mismatch in {fold_dir.name}: {len(meta)} vs {arr["labels"].shape[0]}')
        meta['fold_dir'] = fold_dir.name
        fold_frames.append(meta)
        probs_parts.append(arr['probs'].astype(np.float32))
        logits_parts.append(arr['logits'].astype(np.float32))
        label_parts.append(arr['labels'].astype(np.float32))
    if not fold_frames:
        raise FileNotFoundError(f'Could not find pooled exp_011 fold outputs under {root}')
    meta = pd.concat(fold_frames, ignore_index=True)
    probs = np.concatenate(probs_parts, axis=0)
    logits = np.concatenate(logits_parts, axis=0)
    labels = np.concatenate(label_parts, axis=0)
    return meta, probs, logits, labels


def resolve_v18_artifacts_dir(hint: str | None = None) -> Path | None:
    search_roots = [REPO_ROOT, REPO_ROOT / 'submissions', KAGGLE_INPUT_ROOT]
    found = []
    for root in search_roots:
        if not root.exists():
            continue
        for manifest in root.rglob('artifacts_manifest.json'):
            parent = manifest.parent
            if hint is None or hint.lower() in str(parent).lower():
                found.append(parent)
    if not found:
        return None
    found = sorted(set(found), key=lambda p: len(str(p)))
    return found[0]


taxonomy = pd.read_csv(TAXONOMY_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH, nrows=1)
PRIMARY_LABELS = sample_sub.columns[1:].tolist()
taxonomy = taxonomy.copy()
taxonomy['primary_label'] = taxonomy['primary_label'].astype(str)
taxonomy_lookup = taxonomy.set_index('primary_label')
print('n_classes:', len(PRIMARY_LABELS))


In [ ]:
exp011_meta, exp011_probs, exp011_logits, exp011_labels = load_exp011_pooled(EXP011_ROOT)

exp011_meta['is_soundscape'] = exp011_meta['source'].eq('soundscape_clip')
soundscape_parsed = exp011_meta['filename'].astype(str).apply(parse_soundscape_filename).apply(pd.Series)
for col in soundscape_parsed.columns:
    exp011_meta[col] = soundscape_parsed[col]

exp011_meta['primary_taxon'] = exp011_meta['primary_label'].map(taxonomy_lookup['class_name']).fillna('Unknown')
exp011_meta['scientific_name'] = exp011_meta['primary_label'].map(taxonomy_lookup['scientific_name']).fillna('Unknown')
exp011_meta['label_list'] = exp011_meta['labels'].apply(split_label_string)
exp011_meta['label_count'] = exp011_meta['label_list'].apply(len)
exp011_meta['has_texture_label'] = exp011_meta['label_list'].apply(
    lambda labels: any(taxonomy_lookup['class_name'].get(lbl, 'Unknown') in TEXTURE_TAXA for lbl in labels)
)

sound_mask = exp011_meta['is_soundscape'].to_numpy()
print('pooled rows:', len(exp011_meta))
print('train_audio rows:', int((~sound_mask).sum()))
print('soundscape rows:', int(sound_mask.sum()))
print('fold_dirs:', sorted(exp011_meta['fold_dir'].unique().tolist()))
print('sources:', exp011_meta['source'].value_counts().to_dict())
print('soundscape sites:', exp011_meta.loc[sound_mask, 'site'].value_counts().to_dict())
print('soundscape hours:', exp011_meta.loc[sound_mask, 'hour_utc'].value_counts().sort_index().to_dict())
exp011_meta.head(3)


In [ ]:
classwise_all = safe_classwise_auc(exp011_labels, exp011_probs, PRIMARY_LABELS).rename(
    columns={'auc': 'auc_all', 'positive_rows': 'positive_rows_all', 'negative_rows': 'negative_rows_all', 'mean_prob': 'mean_prob_all'}
)
classwise_sound = safe_classwise_auc(exp011_labels[sound_mask], exp011_probs[sound_mask], PRIMARY_LABELS).rename(
    columns={'auc': 'auc_soundscape', 'positive_rows': 'positive_rows_soundscape', 'negative_rows': 'negative_rows_soundscape', 'mean_prob': 'mean_prob_soundscape'}
)

classwise = classwise_all.merge(
    classwise_sound[['primary_label', 'auc_soundscape', 'positive_rows_soundscape', 'negative_rows_soundscape', 'mean_prob_soundscape', 'scored']],
    on='primary_label',
    how='left',
    suffixes=('', '_soundscape')
)
taxonomy_join_cols = ['primary_label', 'scientific_name', 'class_name']
for optional_col in ['order_name', 'family_name']:
    if optional_col in taxonomy.columns:
        taxonomy_join_cols.append(optional_col)
classwise = classwise.merge(
    taxonomy[taxonomy_join_cols],
    on='primary_label',
    how='left'
)
classwise['is_texture_taxon'] = classwise['class_name'].isin(TEXTURE_TAXA)
classwise['auc_gap_sound_minus_all'] = classwise['auc_soundscape'] - classwise['auc_all']
classwise['scored_soundscape'] = classwise['positive_rows_soundscape'].fillna(0).gt(0) & classwise['negative_rows_soundscape'].fillna(0).gt(0)

macro_all = float(classwise.loc[classwise['scored'], 'auc_all'].mean())
macro_sound = float(classwise.loc[classwise['scored_soundscape'], 'auc_soundscape'].mean())
weak_sound = classwise.loc[classwise['scored_soundscape']].sort_values(['auc_soundscape', 'positive_rows_soundscape']).reset_index(drop=True)
strong_sound = classwise.loc[classwise['scored_soundscape']].sort_values(['auc_soundscape', 'positive_rows_soundscape'], ascending=[False, False]).reset_index(drop=True)

print('macro_auc_all:', round(macro_all, 6))
print('macro_auc_soundscape:', round(macro_sound, 6))
print('\nWeakest soundscape classes:')
print(weak_sound[['primary_label', 'class_name', 'auc_soundscape', 'positive_rows_soundscape', 'auc_gap_sound_minus_all']].head(12).to_string(index=False))
print('\nStrongest soundscape classes:')
print(strong_sound[['primary_label', 'class_name', 'auc_soundscape', 'positive_rows_soundscape', 'auc_gap_sound_minus_all']].head(12).to_string(index=False))


In [ ]:
sound_meta = exp011_meta.loc[sound_mask].reset_index(drop=True)
sound_labels = exp011_labels[sound_mask]
sound_probs = exp011_probs[sound_mask]

taxon_summary = (
    classwise.loc[classwise['scored_soundscape']]
    .groupby('class_name', dropna=False)
    .agg(
        classes=('primary_label', 'count'),
        mean_soundscape_auc=('auc_soundscape', 'mean'),
        median_soundscape_auc=('auc_soundscape', 'median'),
        mean_gap_sound_minus_all=('auc_gap_sound_minus_all', 'mean'),
        total_positive_rows=('positive_rows_soundscape', 'sum'),
    )
    .sort_values('mean_soundscape_auc')
    .reset_index()
)

site_summary = grouped_macro_auc(sound_meta, sound_labels, sound_probs, 'site')
hour_summary = grouped_macro_auc(sound_meta, sound_labels, sound_probs, 'hour_utc')

row_true_conf = (sound_probs * sound_labels).sum(axis=1) / np.clip(sound_labels.sum(axis=1), 1, None)
row_false_conf = (sound_probs * (1.0 - sound_labels)).sum(axis=1) / np.clip((1.0 - sound_labels).sum(axis=1), 1, None)
sound_meta['row_true_conf'] = row_true_conf
sound_meta['row_false_conf'] = row_false_conf
sound_meta['row_margin'] = sound_meta['row_true_conf'] - sound_meta['row_false_conf']

site_conf_summary = (
    sound_meta.groupby('site', dropna=False)
    .agg(
        rows=('filename', 'size'),
        files=('filename', 'nunique'),
        mean_true_conf=('row_true_conf', 'mean'),
        mean_false_conf=('row_false_conf', 'mean'),
        mean_margin=('row_margin', 'mean'),
    )
    .sort_values('mean_margin')
    .reset_index()
)

hour_conf_summary = (
    sound_meta.groupby('hour_utc', dropna=False)
    .agg(
        rows=('filename', 'size'),
        files=('filename', 'nunique'),
        mean_true_conf=('row_true_conf', 'mean'),
        mean_false_conf=('row_false_conf', 'mean'),
        mean_margin=('row_margin', 'mean'),
    )
    .sort_values('mean_margin')
    .reset_index()
)

print('Taxon summary:')
print(taxon_summary.to_string(index=False))
print('\nMost difficult sites by soundscape macro AUC:')
print(site_summary.head(10).to_string(index=False))
print('\nMost difficult hours by soundscape macro AUC:')
print(hour_summary.head(10).to_string(index=False))


In [ ]:
v18_artifacts_dir = resolve_v18_artifacts_dir(V18_ARTIFACTS_HINT)
v18_crosswalk = None
v18_summary = None

if v18_artifacts_dir is None:
    print('No V18 artifacts found locally or attached; skipping exp_015d crosswalk.')
else:
    manifest = json.loads((v18_artifacts_dir / 'artifacts_manifest.json').read_text())
    threshold_path = v18_artifacts_dir / manifest['artifact_files']['thresholds']
    thresholds = np.load(threshold_path).astype(np.float32)

    calibration_manifest = manifest.get('calibration', {})
    calibration_manifest_path = manifest.get('artifact_files', {}).get('calibration_manifest')
    if calibration_manifest_path and (v18_artifacts_dir / calibration_manifest_path).exists():
        calibration_manifest = json.loads((v18_artifacts_dir / calibration_manifest_path).read_text())

    v18_crosswalk = pd.DataFrame({
        'primary_label': manifest['primary_labels'],
        'v18_threshold': thresholds,
    })
    v18_crosswalk['v18_threshold_bucket'] = pd.cut(
        v18_crosswalk['v18_threshold'],
        bins=[-np.inf, 0.35, 0.45, 0.55, 0.65, np.inf],
        labels=['<=0.35', '(0.35,0.45]', '(0.45,0.55]', '(0.55,0.65]', '>0.65'],
    )
    v18_crosswalk = v18_crosswalk.merge(
        classwise[['primary_label', 'class_name', 'auc_all', 'auc_soundscape', 'auc_gap_sound_minus_all']],
        on='primary_label',
        how='left'
    )
    v18_summary = {
        'artifacts_dir': str(v18_artifacts_dir),
        'has_residual': bool(manifest.get('has_residual', False)),
        'proxy_reduce': manifest.get('proxy_reduce'),
        'tta_shifts': manifest.get('tta_shifts'),
        'rank_aware_power': manifest.get('rank_aware_power'),
        'delta_shift_alpha': manifest.get('delta_shift_alpha'),
        'calibration': calibration_manifest,
        'threshold_mean': float(thresholds.mean()),
        'threshold_min': float(thresholds.min()),
        'threshold_max': float(thresholds.max()),
    }
    print('V18 artifacts dir:', v18_artifacts_dir)
    print('Threshold stats:', {k: v18_summary[k] for k in ['threshold_mean', 'threshold_min', 'threshold_max']})
    print('Lowest native soundscape AUC classes with aggressive V18 thresholds:')
    print(
        v18_crosswalk.sort_values(['auc_soundscape', 'v18_threshold']).head(12)[['primary_label', 'class_name', 'auc_soundscape', 'v18_threshold', 'v18_threshold_bucket']]
        .to_string(index=False)
    )


In [ ]:
weak_sound_path = OUTPUT_DIR / 'exp011_weak_soundscape_classes.csv'
classwise_path = OUTPUT_DIR / 'exp011_classwise_auc.csv'
taxon_path = OUTPUT_DIR / 'exp011_taxon_summary.csv'
site_auc_path = OUTPUT_DIR / 'exp011_soundscape_site_macro_auc.csv'
site_conf_path = OUTPUT_DIR / 'exp011_soundscape_site_confidence.csv'
hour_auc_path = OUTPUT_DIR / 'exp011_soundscape_hour_macro_auc.csv'
hour_conf_path = OUTPUT_DIR / 'exp011_soundscape_hour_confidence.csv'
snapshot_path = OUTPUT_DIR / 'report_snapshot.json'

classwise.to_csv(classwise_path, index=False)
weak_sound.to_csv(weak_sound_path, index=False)
taxon_summary.to_csv(taxon_path, index=False)
site_summary.to_csv(site_auc_path, index=False)
site_conf_summary.to_csv(site_conf_path, index=False)
hour_summary.to_csv(hour_auc_path, index=False)
hour_conf_summary.to_csv(hour_conf_path, index=False)

artifact_crosswalk_path = None
artifact_summary_path = None
if v18_crosswalk is not None:
    artifact_crosswalk_path = OUTPUT_DIR / 'v18_artifact_classwise_crosswalk.csv'
    artifact_summary_path = OUTPUT_DIR / 'v18_artifact_summary.json'
    v18_crosswalk.to_csv(artifact_crosswalk_path, index=False)
    artifact_summary_path.write_text(json.dumps(v18_summary, indent=2))

report_snapshot = {
    'experiment_id': 'exp_017',
    'experiment_name': 'v18_error_report',
    'rows_total': int(len(exp011_meta)),
    'rows_soundscape': int(sound_mask.sum()),
    'rows_train_audio': int((~sound_mask).sum()),
    'macro_auc_all': macro_all,
    'macro_auc_soundscape': macro_sound,
    'scored_classes_all': int(classwise['scored'].sum()),
    'scored_classes_soundscape': int(classwise['scored_soundscape'].sum()),
    'output_dir': str(OUTPUT_DIR),
    'v18_artifacts_found': v18_artifacts_dir is not None,
    'v18_artifacts_dir': str(v18_artifacts_dir) if v18_artifacts_dir is not None else None,
    'files': {
        'classwise_auc': str(classwise_path),
        'weak_soundscape_classes': str(weak_sound_path),
        'taxon_summary': str(taxon_path),
        'site_macro_auc': str(site_auc_path),
        'site_confidence': str(site_conf_path),
        'hour_macro_auc': str(hour_auc_path),
        'hour_confidence': str(hour_conf_path),
        'v18_artifact_classwise_crosswalk': str(artifact_crosswalk_path) if artifact_crosswalk_path else None,
        'v18_artifact_summary': str(artifact_summary_path) if artifact_summary_path else None,
    },
}
snapshot_path.write_text(json.dumps(report_snapshot, indent=2))

print('Saved report snapshot to:', snapshot_path)
print(json.dumps(report_snapshot, indent=2))


## Next steps

- use `exp011_weak_soundscape_classes.csv` to identify the most plausible specialist or threshold-sensitive candidates
- if a V18 artifact dataset is attached, inspect `v18_artifact_classwise_crosswalk.csv` for classes where aggressive V18 thresholds overlap with weak native performance
- if needed, extend this notebook later with a true `exp_015d` prediction replay once local or attached prediction artifacts are available
